In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers and sample data
data = data.replace("?", np.nan)
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

# Fill missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare features, target, and metadata
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

# Set experiment constants and seeds
N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Initialize result containers
scores = {}
synthetic_datasets = {}
quality_results = []


In [5]:
# Single run

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train/test split without leakage

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)



================ SINGLE RUN ================
Training TabDDPM...
[0]
87
{'num_classes': 4, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(87)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.3014 Sum: 0.3014
Step 1000/1000 MLoss: 0.0 GLoss: 0.2539 Sum: 0.2539
mlp
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Discrete cols: [2, 3, 4]
Num shape:  (10000, 6)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 30.11it/s]|
Column Shapes Score: 24.23%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 512.26it/s]|
Column Pair Trends Score: 0.0%

Overall Score (Average): 12.11%

TabDDPM: 0.1211


In [6]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()


Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:03<00:00,  4.62it/s]|
Column Shapes Score: 54.29%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 396.85it/s]|
Column Pair Trends Score: 14.14%

Overall Score (Average): 34.22%

ForestDiffusion: 0.3422


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            # Extract X and y from the full training DataFrame for the current seed
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            # Extract X and y from the full testing DataFrame for the current seed
            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            # Determine if stratification is possible for y_train_full
            stratify_y_train_full = y_train_full if y_train_full.value_counts().min() >= 2 else None
            if stratify_y_train_full is None:
                print(
                    f"Warning: Cannot stratify training data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `train_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the first train-test split (from train_df to get actual training set for classifier)
            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_train_full
            )

            # Determine if stratification is possible for y_test_full
            # This is usually for real data and should be fine, but included for robustness.
            stratify_y_test_full = y_test_full if y_test_full.value_counts().min() >= 2 else None
            if stratify_y_test_full is None:
                print(
                    f"Warning: Cannot stratify testing data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `test_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the second train-test split (from test_df to get actual testing set for classifier)
            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_test_full
            )

            scaler = StandardScaler().fit(X_train_split)

            X_train_s = scaler.transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train_split)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test_split, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1)

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1)

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1)

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1)

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy \u00b1 SD": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 \u00b1 SD": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision \u00b1 SD": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall \u00b1 SD": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}",

            "Accuracy (Mean\u00b1Std)": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 (Mean\u00b1Std)": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision (Mean\u00b1Std)": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall (Mean\u00b1Std)": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [18]:
# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers
data = data.replace("?", np.nan)

# Clean income target into two classes only
data[target_col] = (
    data[target_col]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
)

print(data[target_col].value_counts())

# Take 1000 real samples
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

# Fill missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare processed dataset
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

print("\nEncoded income classes:")
print(processed_data[target_col].value_counts())
print(label_encoders[target_col].classes_)


income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Encoded income classes:
income
0    775
1    225
Name: count, dtype: int64
['<=50K' '>50K']


In [19]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col   # Adult dataset target column: income

model_order = ["TabDDPM", "ForestDiffusion"]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label_col=target_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy \u00b1 SD",
            "F1 \u00b1 SD",
            "Precision \u00b1 SD",
            "Recall \u00b1 SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    # Align synthetic columns with processed_data integer encoding.
    for col, le in label_encoders.items():
        if col not in synthetic_train_df.columns:
            continue
        vals = synthetic_train_df[col]
        if pd.api.types.is_integer_dtype(vals):
            continue
        str_vals = vals.astype(str).str.strip()
        known = set(le.classes_)
        if set(str_vals.unique()).issubset(known):
            synthetic_train_df[col] = le.transform(str_vals)
        else:
            synthetic_train_df[col] = (
                pd.to_numeric(vals, errors="coerce")
                .fillna(0)
                .round()
                .astype(int)
                .clip(0, len(le.classes_) - 1)
            )

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label_col=target_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy \u00b1 SD",
                "F1 \u00b1 SD",
                "Precision \u00b1 SD",
                "Recall \u00b1 SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=('_TRTR', '_TSTR')
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy \u00b1 SD_TRTR",
                "Accuracy \u00b1 SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)

TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
5,RandomForest,0.8385 ± 0.0258,0.8306 ± 0.0285,0.8293 ± 0.0298,0.8385 ± 0.0258
8,AdaBoost,0.8370 ± 0.0234,0.8271 ± 0.0270,0.8275 ± 0.0280,0.8370 ± 0.0234
1,SVM-RBF,0.8335 ± 0.0183,0.8100 ± 0.0257,0.8244 ± 0.0226,0.8335 ± 0.0183
7,GradientBoost,0.8330 ± 0.0241,0.8246 ± 0.0253,0.8239 ± 0.0264,0.8330 ± 0.0241
0,LogReg,0.8280 ± 0.0218,0.8104 ± 0.0263,0.8146 ± 0.0279,0.8280 ± 0.0218
6,ExtraTrees,0.8240 ± 0.0247,0.8177 ± 0.0258,0.8164 ± 0.0265,0.8240 ± 0.0247
3,NaiveBayes,0.8090 ± 0.0258,0.7857 ± 0.0299,0.7898 ± 0.0386,0.8090 ± 0.0258
9,MLP,0.8075 ± 0.0255,0.8070 ± 0.0238,0.8083 ± 0.0222,0.8075 ± 0.0255
2,KNN,0.8070 ± 0.0225,0.8009 ± 0.0251,0.7978 ± 0.0266,0.8070 ± 0.0225
4,DecisionTree,0.7840 ± 0.0291,0.7853 ± 0.0286,0.7881 ± 0.0289,0.7840 ± 0.0291


TabDDPM - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
2,KNN,0.6030 ± 0.0649,0.6280 ± 0.0553,0.6854 ± 0.0242,0.6030 ± 0.0649
6,ExtraTrees,0.5955 ± 0.0873,0.6040 ± 0.0570,0.6366 ± 0.0158,0.5955 ± 0.0873
3,NaiveBayes,0.5655 ± 0.1774,0.5443 ± 0.1772,0.6655 ± 0.0927,0.5655 ± 0.1774
4,DecisionTree,0.5330 ± 0.0894,0.5585 ± 0.0776,0.6609 ± 0.0734,0.5330 ± 0.0894
8,AdaBoost,0.5305 ± 0.1057,0.5489 ± 0.0756,0.6272 ± 0.0574,0.5305 ± 0.1057
5,RandomForest,0.5125 ± 0.1124,0.5395 ± 0.1035,0.6430 ± 0.0431,0.5125 ± 0.1124
9,MLP,0.4575 ± 0.2712,0.3571 ± 0.2926,0.4823 ± 0.2529,0.4575 ± 0.2712
7,GradientBoost,0.3895 ± 0.1069,0.4091 ± 0.1220,0.6031 ± 0.1038,0.3895 ± 0.1069
1,SVM-RBF,0.2305 ± 0.0083,0.0973 ± 0.0146,0.5673 ± 0.3169,0.2305 ± 0.0083
0,LogReg,0.2305 ± 0.0083,0.0973 ± 0.0146,0.5673 ± 0.3169,0.2305 ± 0.0083


TabDDPM - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
7,GradientBoost,0.7785 ± 0.0389,0.7931 ± 0.0347,0.8349 ± 0.0249,0.7785 ± 0.0389
3,NaiveBayes,0.7725 ± 0.0282,0.7666 ± 0.0281,0.7633 ± 0.0292,0.7725 ± 0.0282
5,RandomForest,0.7465 ± 0.0207,0.7645 ± 0.0189,0.8155 ± 0.0210,0.7465 ± 0.0207
8,AdaBoost,0.7280 ± 0.0322,0.7496 ± 0.0290,0.8275 ± 0.0238,0.7280 ± 0.0322
0,LogReg,0.7205 ± 0.0252,0.7426 ± 0.0226,0.8169 ± 0.0142,0.7205 ± 0.0252
6,ExtraTrees,0.7175 ± 0.0236,0.7399 ± 0.0215,0.8160 ± 0.0206,0.7175 ± 0.0236
1,SVM-RBF,0.7155 ± 0.0228,0.7383 ± 0.0206,0.8183 ± 0.0154,0.7155 ± 0.0228
9,MLP,0.7040 ± 0.0368,0.7272 ± 0.0331,0.8053 ± 0.0197,0.7040 ± 0.0368
2,KNN,0.6820 ± 0.0204,0.7079 ± 0.0188,0.8002 ± 0.0193,0.6820 ± 0.0204
4,DecisionTree,0.6325 ± 0.0410,0.6618 ± 0.0371,0.7511 ± 0.0243,0.6325 ± 0.0410


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,ForestDiffusion,RandomForest,0.0920,0.066090,0.013755,0.0920,0.8385 ± 0.0258,0.7465 ± 0.0207
1,ForestDiffusion,AdaBoost,0.1090,0.077539,-0.000012,0.1090,0.8370 ± 0.0234,0.7280 ± 0.0322
2,ForestDiffusion,SVM-RBF,0.1180,0.071669,0.006083,0.1180,0.8335 ± 0.0183,0.7155 ± 0.0228
3,ForestDiffusion,GradientBoost,0.0545,0.031486,-0.011005,0.0545,0.8330 ± 0.0241,0.7785 ± 0.0389
4,ForestDiffusion,LogReg,0.1075,0.067821,-0.002271,0.1075,0.8280 ± 0.0218,0.7205 ± 0.0252
5,ForestDiffusion,ExtraTrees,0.1065,0.077824,0.000383,0.1065,0.8240 ± 0.0247,0.7175 ± 0.0236
6,ForestDiffusion,NaiveBayes,0.0365,0.019127,0.026498,0.0365,0.8090 ± 0.0258,0.7725 ± 0.0282
7,ForestDiffusion,MLP,0.1035,0.079721,0.003005,0.1035,0.8075 ± 0.0255,0.7040 ± 0.0368
8,ForestDiffusion,KNN,0.1250,0.092960,-0.002384,0.1250,0.8070 ± 0.0225,0.6820 ± 0.0204
9,ForestDiffusion,DecisionTree,0.1515,0.123450,0.036981,0.1515,0.7840 ± 0.0291,0.6325 ± 0.0410


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,0.10040,0.070769,0.007103,0.10040
1,TabDDPM,0.35535,0.371514,0.198149,0.35535


In [20]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
